In [2]:
import matplotlib.pyplot as plt
import numpy as np
import math
import ipywidgets as widgets
from IPython.display import display


sinus = True

def sigmoid(x):
	return 1. / (1. + np.exp(-x))

def get_ds(x, x_max, switches):
	thresholds = [x_max * s / 100.0 for s in switches]
	flips = sum(x >= t for t in thresholds)
	return -1 if (flips % 2 == 0) else 1

def plot(clm_w, hgm_w1, hgm_w2, _):
	switches = (12, 30, 45, 50, 67, 81, 90, 95)
	step = 0.01
	x_max = 8. * math.pi
	xs = np.arange(0, x_max, step)

	clm_ts = np.array([(math.sin(x) if sinus else 1) * clm_w for x in xs])
	hgm_ts = np.array([(math.sin(x) if sinus else 1) * hgm_w1 * sigmoid(get_ds(x, x_max, switches) * hgm_w2) for x in xs])

	fig = plt.figure(figsize=(16, 4))
	ax = plt.axes()
	ax.set_xlim(0, x_max)
	ax.set_ylim(-2, 2)

	thresholds = [x_max * s / 100.0 for s in switches]
	spans = [(0, thresholds[0]), (thresholds[1], thresholds[2]), (thresholds[3], thresholds[4]), (thresholds[5], thresholds[6]), (thresholds[7], x_max)]
	for (a, b) in spans: ax.axvspan(a, b, color='0.9', alpha=0.5, lw=0)
	
	ax.plot(xs, clm_ts, alpha=0.75, color='green')
	ax.plot(xs, hgm_ts, alpha=0.75, color='blue')

	plt.show()


clm_w_slider = widgets.FloatSlider(min=0., max=2., step=.01, value=1., description='clm w')
hgm_w1_slider = widgets.FloatSlider(min=0., max=4., step=.01, value=2., description='hgm w 1')
hgm_w2_slider = widgets.FloatSlider(min=0., max=10., step=.01, value=0., description='hgm w 2')

gating_slider = widgets.FloatSlider(min=1e-9, max=1., step=.01, value=1., description='gating')
def derive_hgm_params(_=None):
	g = float(gating_slider.value)
	clm_w = float(clm_w_slider.value)

	hgm_w1_slider.value = (1.0 + g) * clm_w
	hgm_w2_slider.value = -math.log(g)

	#hgm_w1_slider.value = clm_w + g
	#hgm_w2_slider.value = math.log(clm_w / g)

    
gating_slider.observe(derive_hgm_params, names='value')
#clm_w_slider.observe(derive_hgm_params, names='value')

gating_label = widgets.Label(value=f'(inferred gating: {1.})')
def derive_gating_param(_=None):
	clm_w = float(clm_w_slider.value)

	hgm_w1 = float(hgm_w1_slider.value)
	hgm_w2 = float(hgm_w2_slider.value)

	gating = hgm_w1 * sigmoid(-hgm_w2) / clm_w
	gating_label.value = f'(inferred gating: {gating:.2f})'
hgm_w2_slider.observe(derive_gating_param, names='value')

check = widgets.Checkbox(value=False, description='constant ews', indent=True)
def on_change(change):
	global sinus
	sinus = not change['new']
check.observe(on_change, names='value')


row0 = widgets.HBox([clm_w_slider, check])
row1 = widgets.HBox([hgm_w1_slider, hgm_w2_slider, gating_slider, gating_label])
ui = widgets.VBox([row0, row1])
out = widgets.interactive_output(
	plot,
	{
		'clm_w':  clm_w_slider,
		'hgm_w1': hgm_w1_slider,
		'hgm_w2': hgm_w2_slider,
		'_':	  check,
	}
)
display(ui, out)

Output()